In [1]:
import pandas as pd
import os

In [15]:
# Folder where all CSVs are stored
data_folder = "../data/raw"

# Generate file list (2006–2018)
files = [f"DUQ_LOAD_HOURLY_{year}.csv" for year in range(2006, 2017)]

file_paths = [os.path.join(data_folder, f) for f in files]

file_paths

['D:\\Downloads\\BTP\\model_training\\lstm_training\\DUQ_LOAD_HOURLY_2006.csv',
 'D:\\Downloads\\BTP\\model_training\\lstm_training\\DUQ_LOAD_HOURLY_2007.csv',
 'D:\\Downloads\\BTP\\model_training\\lstm_training\\DUQ_LOAD_HOURLY_2008.csv',
 'D:\\Downloads\\BTP\\model_training\\lstm_training\\DUQ_LOAD_HOURLY_2009.csv',
 'D:\\Downloads\\BTP\\model_training\\lstm_training\\DUQ_LOAD_HOURLY_2010.csv',
 'D:\\Downloads\\BTP\\model_training\\lstm_training\\DUQ_LOAD_HOURLY_2011.csv',
 'D:\\Downloads\\BTP\\model_training\\lstm_training\\DUQ_LOAD_HOURLY_2012.csv',
 'D:\\Downloads\\BTP\\model_training\\lstm_training\\DUQ_LOAD_HOURLY_2013.csv',
 'D:\\Downloads\\BTP\\model_training\\lstm_training\\DUQ_LOAD_HOURLY_2014.csv',
 'D:\\Downloads\\BTP\\model_training\\lstm_training\\DUQ_LOAD_HOURLY_2015.csv',
 'D:\\Downloads\\BTP\\model_training\\lstm_training\\DUQ_LOAD_HOURLY_2016.csv']

In [16]:
dfs = []

cols_to_drop = [
    'nerc_region',
    'mkt_region',
    'zone',
    'load_area',
    'is_verified'
]

for file in file_paths:
    print(f"\nReading {file}")
    
    df = pd.read_csv(file)
    
    # Normalize column names
    df.columns = [col.strip().lower() for col in df.columns]
    
    print("Columns found:", df.columns.tolist())  # DEBUG
    
    # 🔍 Find datetime column dynamically
    datetime_col = None
    
    for col in df.columns:
        if 'datetime' in col:
            datetime_col = col
            break
    
    if datetime_col is None:
        raise ValueError(f"No datetime column found in {file}")
    
    print(f"Using datetime column: {datetime_col}")
    
    # Convert datetime
    df['datetime'] = pd.to_datetime(
        df[datetime_col],
        format='%m/%d/%Y %I:%M:%S %p',
        errors='coerce'
    )
    
    # Drop original datetime column
    df.drop(columns=[datetime_col], inplace=True)
    
    # Drop unwanted columns safely
    df.drop(columns=[col for col in cols_to_drop if col in df.columns], inplace=True)
    
    # Remove invalid datetime rows
    df = df.dropna(subset=['datetime'])
    
    dfs.append(df)


Reading D:\Downloads\BTP\model_training\lstm_training\DUQ_LOAD_HOURLY_2006.csv
Columns found: ['datetime_beginning_utc', 'datetime_beginning_ept', 'nerc_region', 'mkt_region', 'zone', 'load_area', 'mw', 'is_verified']
Using datetime column: datetime_beginning_utc

Reading D:\Downloads\BTP\model_training\lstm_training\DUQ_LOAD_HOURLY_2007.csv
Columns found: ['datetime_beginning_utc', 'datetime_beginning_ept', 'nerc_region', 'mkt_region', 'zone', 'load_area', 'mw', 'is_verified']
Using datetime column: datetime_beginning_utc

Reading D:\Downloads\BTP\model_training\lstm_training\DUQ_LOAD_HOURLY_2008.csv
Columns found: ['datetime_beginning_utc', 'datetime_beginning_ept', 'nerc_region', 'mkt_region', 'zone', 'load_area', 'mw', 'is_verified']
Using datetime column: datetime_beginning_utc

Reading D:\Downloads\BTP\model_training\lstm_training\DUQ_LOAD_HOURLY_2009.csv
Columns found: ['datetime_beginning_utc', 'datetime_beginning_ept', 'nerc_region', 'mkt_region', 'zone', 'load_area', 'mw', '

In [17]:
import pandas as pd
import os

# 📂 Input folder (where your original CSVs are)
input_folder = r"../data/raw"

# 📂 Output folder (cleaned files will be saved here)
output_folder = "../data/interim/cleaned_data_DUQ"

# Create folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Get all CSV files
files = [f for f in os.listdir(input_folder) if f.endswith(".csv")]

cols_to_drop = [
    'datetime_beginning_ept',
    'nerc_region',
    'mkt_region',
    'zone',
    'load_area',
    'is_verified'
]

for file in files:
    print(f"Processing {file}")
    
    file_path = os.path.join(input_folder, file)
    df = pd.read_csv(file_path)
    
    # Normalize column names
    df.columns = [col.strip().lower() for col in df.columns]
    
    # Drop columns safely (only if they exist)
    df.drop(columns=[col for col in cols_to_drop if col in df.columns], inplace=True)
    
    # Save cleaned file
    output_path = os.path.join(output_folder, file)
    df.to_csv(output_path, index=False)

print("\n✅ All files cleaned and saved in:", output_folder)

Processing DUQ_LOAD_HOURLY_2006.csv
Processing DUQ_LOAD_HOURLY_2007.csv
Processing DUQ_LOAD_HOURLY_2008.csv
Processing DUQ_LOAD_HOURLY_2009.csv
Processing DUQ_LOAD_HOURLY_2010.csv
Processing DUQ_LOAD_HOURLY_2011.csv
Processing DUQ_LOAD_HOURLY_2012.csv
Processing DUQ_LOAD_HOURLY_2013.csv
Processing DUQ_LOAD_HOURLY_2014.csv
Processing DUQ_LOAD_HOURLY_2015.csv
Processing DUQ_LOAD_HOURLY_2016.csv

✅ All files cleaned and saved in: D:\Downloads\BTP\model_training\lstm_training\cleaned_data_DUQ


In [20]:
##MERGING

In [28]:
input_folder = r"../data/interim/cleaned_data_DUQ"

files = [f for f in os.listdir(input_folder) if f.endswith(".csv")]

print("Total files found:", len(files))

Total files found: 11


In [29]:
dfs = []

for file in sorted(files):  # sorted ensures year order (important)
    print(f"Reading {file}")
    
    file_path = os.path.join(input_folder, file)
    df = pd.read_csv(file_path)
    
    # Normalize column names
    df.columns = [col.strip().lower() for col in df.columns]
    
    # 🔍 Detect timestamp column
    datetime_col = None
    for col in df.columns:
        if 'datetime' in col:
            datetime_col = col
            break
    
    if datetime_col is None:
        raise ValueError(f"No datetime column found in {file}")
    
    # Convert datetime (use known format)
    df['datetime'] = pd.to_datetime(
        df[datetime_col],
        format='%m/%d/%Y %I:%M:%S %p',
        errors='coerce'
    )
    
    # Drop original datetime column
    df.drop(columns=[datetime_col], inplace=True)
    
    # Drop invalid rows
    df = df.dropna(subset=['datetime'])
    
    dfs.append(df)

Reading DUQ_LOAD_HOURLY_2006.csv
Reading DUQ_LOAD_HOURLY_2007.csv
Reading DUQ_LOAD_HOURLY_2008.csv
Reading DUQ_LOAD_HOURLY_2009.csv
Reading DUQ_LOAD_HOURLY_2010.csv
Reading DUQ_LOAD_HOURLY_2011.csv
Reading DUQ_LOAD_HOURLY_2012.csv
Reading DUQ_LOAD_HOURLY_2013.csv
Reading DUQ_LOAD_HOURLY_2014.csv
Reading DUQ_LOAD_HOURLY_2015.csv
Reading DUQ_LOAD_HOURLY_2016.csv


In [30]:
combined_df = pd.concat(dfs, ignore_index=True)

print("Rows before deduplication:", len(combined_df))

Rows before deduplication: 87682


In [31]:
# Sort BEFORE removing duplicates
combined_df.sort_values(by='datetime', inplace=True)

# Remove duplicates (keep first occurrence)
combined_df = combined_df.drop_duplicates(subset='datetime', keep='first')

print("Rows after deduplication:", len(combined_df))

Rows after deduplication: 87682


In [33]:
combined_df = combined_df.sort_values(by='datetime').reset_index(drop=True)

In [34]:
output_path = "../data/processed/FINAL_MERGED_DUQ_LOAD_2006_2016.csv"

combined_df.to_csv(output_path, index=False)

print("✅ Final merged file saved at:", output_path)

✅ Final merged file saved at: D:\Downloads\BTP\model_training\lstm_training\cleaned_data_DUQ\FINAL_MERGED_DUQ_LOAD_2006_2016.csv
